# 01 - Phase 1: behavioral faithfulness (Week 1)\n\nSteps 2-5: sample 400 MedQA questions, generate CoT traces, build evidence/control perturbations, score them, compute F(x) + flip rates + Wilcoxon, and select the 100-question mechanistic subset.\n\n**Runtime on a T4 GPU:** a few hours for 400 questions x 3 runs. Results checkpoint after every question, so you can stop/resume safely (re-running skips finished ids).

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


In [ ]:
from src import pipeline
from src.common import load_config
cfg = load_config()
# Override the model here if you want a different candidate:
# cfg["model"]["primary"] = "akshayballal/Qwen3-1.7B-Pubmed-16bit-GRPO"
stats = pipeline.run_phase1(cfg)
print("PHASE 1 DONE")
import json; print(json.dumps(stats, indent=2))


In [ ]:
# @title Download outputs (run after the phase finishes)
import os, zipfile, glob
from google.colab import files
out_dir = "outputs"
zip_path = "/content/outputs.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(os.path.join(out_dir, "**", "*"), recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, "/content"))
files.download(zip_path)
print("Downloaded", zip_path)
